# GASP parameter-sharing experiment: design and local validation
    Start here. This notebook runs a **synthetic smoke test**, not a paper reproduction.
    It uses a real, randomly initialized 11,424-parameter Qwen2 causal transformer and
    real sampling, GRPO, repair NLL, backward passes and AdamW updates. Its binary
    vocabulary makes the complete pipeline affordable on CPU. No weights download
    or paid resource is required after dependencies are installed.

    Open via Colab **File → Upload notebook**, then provide `dist/gasp-colab.zip`
    to the bootstrap cell. Full experiments are never launched from this notebook.


# Implementation contract after fidelity review

This supersedes the original plan's custom masking, cached-trace fallback, local
evaluation generator and role-specific optimizer states for shared parameters.
Reference: [Li et al., arXiv:2602.00173v1](https://arxiv.org/html/2602.00173v1),
Sections 2–4 and Appendices A, B and E. See EXPERIMENT.md for unresolved details.

1. Filter 7,500 MATH training candidates using four initial-model solutions at
   temperature 0.7; retain questions only when all final answers are correct.
2. Initialize the agent and polluter from the same checkpoint. Unified aliases
   their parameters; split copies them once and never resynchronizes them.
3. Alternate five agent and five polluter optimizer updates. Unified freezes a
   copy of the opposing role at block entry. Split uses the inactive live model
   under no-grad/eval and allocates no third model.
4. Draw a correct current-agent clean trace with rejection sampling. Stop after
   the configured retry limit if none succeeds; never fall back to an old trace.
   Choose alpha from {0, .25, .5, .75} and take the next ~10% of reasoning tokens,
   subject to the clean-window cap. Preserve boundaries when injecting text.
5. Use group-normalized binary correctness for agent GRPO and 1-correctness for
   polluter GRPO. Include every candidate in normalization. Prompt constraints
   do not become extra reward masks, arithmetic rules, or sentence-count filters.
6. Generate the repair with the current agent using both clean and polluted
   windows. Clone the visible repair under the same deployment prompt as agent
   GRPO. Use the token-sum NLL; an empty visible repair contributes an empty sum.
   Do not reject repairs for length, symbolic content, or a boxed final answer.
7. Use one AdamW state per unique parameter and a global cosine/warmup schedule.
   All inactive split gradients stay None, so their weights and moments are not
   updated. Save enough state for exact mid-block CPU resume.
8. Require an independently supplied held-out GPT-5 recoverability corpus and its
   provenance. Never create substitute corruptions. Use identical fixed contexts
   for paired runs, report clean-solved recoverability as primary, and disclose
   subset-selection details. Report full-cache metrics separately.
9. Test real gradient updates, role isolation, resume, clean retries/exhaustion,
   unformatted corruption learning, text boundaries, symbolic rewards and corpus
   import. Keep the offline smoke explicitly separate from target-model evidence.

The full and short-run pilot profiles use full tuning, G=64 and 1024-token limits.
The pilot only reduces update count and reporting intervals. Notebooks default to
full settings and never provision infrastructure. Existing pre-fix outputs are
historical; start new runs and rebuild caches under the revised code.


In [1]:
from pathlib import Path
import os, sys, json, subprocess, zipfile

candidates = [Path(os.environ.get("GASP_ROOT", ".")), Path.cwd(), Path.cwd().parent, Path("/content/gasp")]
ROOT = next((p.resolve() for p in candidates if (p / "src/gasp").is_dir()), None)
if ROOT is None:
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError("Unzip dist/gasp-colab.zip and open this notebook inside the project.")
    print("Select the delivered gasp-colab.zip archive.")
    uploaded = files.upload()
    archives = [p for p in uploaded if p.endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("Select exactly one project archive")
    with zipfile.ZipFile(archives[0]) as z:
        for name in z.namelist():
            if not name.startswith("gasp/") or ".." in Path(name).parts:
                raise ValueError("Unexpected archive member")
        z.extractall("/content")
    ROOT = Path("/content/gasp")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

def command(*args):
    result = subprocess.run([sys.executable, *map(str,args)], cwd=ROOT,
                            env={**os.environ,"PYTHONPATH":str(ROOT/"src")},
                            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    result.check_returncode()

print("Project:", ROOT)


Project: /Users/sidvenkatayogi/Documents/Code/gasp


In [ ]:
# Run once per new Colab runtime. Restart the runtime afterward, then rerun bootstrap.
INSTALL_DEPENDENCIES = True
if INSTALL_DEPENDENCIES:
    assert (3,11) <= sys.version_info[:2] < (3,14), "Use Python 3.11–3.13"
    # These optional image/audio packages can conflict with a pinned text-only torch.
    command("-m", "pip", "uninstall", "-y", "torchvision", "torchaudio")
    command("-m", "pip", "install", "-r", "requirements-colab.txt")
    command("-m", "pip", "install", "--no-deps", ".")
    print("Restart runtime if packages were already imported. No training has started.")


In [2]:
import torch, transformers
torch.set_num_threads(1)
print({"torch":torch.__version__, "transformers":transformers.__version__,
       "cuda":torch.cuda.is_available()})
command("-m", "pytest", "-q")


{'torch': '2.7.1', 'transformers': '4.57.6', 'cuda': False}
........................................                                 [100%]
40 passed in 10.98s



In [3]:
import tempfile
# New output version on every execution; immutable caches cannot be overwritten.
smoke_root = Path(tempfile.mkdtemp(prefix="notebook-smoke-", dir=ROOT/"artifacts"))
command("-m", "gasp", "smoke", "--output", smoke_root)
report = json.loads((smoke_root/"comparison/summary.json").read_text())
print("Synthetic metrics only:", json.dumps(report["summary"],indent=2))
print("Saved:", smoke_root)


step=1 role=agent grad_norm=5.0368
step=2 role=agent grad_norm=3.6136
step=3 role=polluter grad_norm=0.74148
step=4 role=polluter grad_norm=0.95876
step=5 role=agent grad_norm=4.1887
step=6 role=agent grad_norm=4.6754
step=7 role=polluter grad_norm=2.9732
step=8 role=polluter grad_norm=0.58372
step=1 role=agent grad_norm=5.0368
step=2 role=agent grad_norm=3.6136
step=3 role=polluter grad_norm=0.73349
step=4 role=polluter grad_norm=2.4816e-07
step=5 role=agent grad_norm=4.2834
step=6 role=agent grad_norm=4.7024
step=7 role=polluter grad_norm=2.9838
step=8 role=polluter grad_norm=0.61825
{
  "clean_full": {
    "unified": {
      "n": 1,
      "mean": 0.5,
      "sample_std": null
    },
    "split": {
      "n": 1,
      "mean": 0.5,
      "sample_std": null
    },
    "split_minus_unified": {
      "n": 1,
      "mean": 0.0,
      "sample_std": null
    }
  },
  "corrupted_full": {
    "unified": {
      "n": 1,
      "mean": 0.5,
      "sample_std": null
    },
    "split": {
      "n

In [4]:
# Explicit resume at an interior block position; total schedule is unchanged.
from gasp.config import Config
from gasp.trainer import Trainer
cfg = Config.load(ROOT/"configs/smoke.json").with_changes(condition="split")
resumed_dir = smoke_root/"resume-demo"
t = Trainer(cfg, smoke_root/"data/train.jsonl", smoke_root/"data/eval.jsonl", resumed_dir)
t.run(stop_after=3)
checkpoint = t.output/"checkpoint.pt"
del t
t = Trainer(cfg, smoke_root/"data/train.jsonl", smoke_root/"data/eval.jsonl", resumed_dir)
t.resume(checkpoint)
assert t.global_step == 3 and t.role == "polluter" and t.block_position == 1
t.run()
print("Resume complete:", t.role_counts)
del t


step=1 role=agent grad_norm=5.0368
step=2 role=agent grad_norm=3.6136
step=3 role=polluter grad_norm=0.73349
step=4 role=polluter grad_norm=2.4816e-07
step=5 role=agent grad_norm=4.2834
step=6 role=agent grad_norm=4.7024
step=7 role=polluter grad_norm=2.9838
step=8 role=polluter grad_norm=0.61825
Resume complete: {'agent': 4, 'polluter': 4}


## What this does and does not establish
    Passing verifies parameter ownership, frozen snapshots, detached opponents,
    guidance conditioning, cache identity and resume mechanics. It establishes no
    mathematical reasoning or robustness advantage. Continue to `01_prepare_data`
    for full 1.5B tuning on a sufficiently large bf16-capable GPU (plan for 80GB).
